# System Design, Plainly — Part 8: Replication (companion notebook)

Every number in the post is recomputed here from scratch. All four simulations are
toy models on this machine (fixed seeds) — real measurements of a model, not
production telemetry. Run all cells top to bottom; every code cell ends with asserts.

In [ ]:
import random
import numpy as np

rng = random.Random(42)
np_rng = np.random.default_rng(42)
print("seeds fixed: random(42), numpy(42)")

In [ ]:
# Cell 1 — async vs semi-sync vs sync: what the write pays vs what the read risks.
# Model: local commit 2 ms; replica ack RTT 8 ms (same region);
# async apply lag ~ lognormal (median 120 ms, sigma 0.9) under load.
N = 20_000
local_ms, ack_ms = 2.0, 8.0
lag_ms = np_rng.lognormal(mean=np.log(120), sigma=0.9, size=N)
async_p50, async_p99 = np.percentile(lag_ms, [50, 99])
async_lat = local_ms
semisync_lat = local_ms + ack_ms
sync_lat = local_ms + 2 * ack_ms
print(f"async write {async_lat:.0f} ms, staleness p50 {async_p50:.0f} ms / p99 {async_p99:.0f} ms")
print(f"semi-sync write {semisync_lat:.0f} ms, staleness on acked replica 0 ms")
print(f"sync(2) write {sync_lat:.0f} ms, staleness 0 ms")
assert async_lat == 2.0 and semisync_lat == 10.0 and sync_lat == 18.0
assert 100 < async_p50 < 140 and 800 < async_p99 < 1200

In [ ]:
# Cell 2 — quorum math, N=3. A replica that has not yet received a write is
# "behind" with probability b=0.4 (async lag window). 20,000 simulated reads per (W,R).
def stale_rate(W, R, trials=20_000, b=0.4):
    stale = 0
    for _ in range(trials):
        written = set(rng.sample(range(3), W))
        read = set(rng.sample(range(3), R))
        if any(r in written for r in read):
            continue  # a read replica holds the write: fresh
        if all(rng.random() < b for _ in read):
            stale += 1
    return stale / trials * 100

r11 = stale_rate(1, 1); r21 = stale_rate(2, 1)
r12 = stale_rate(1, 2); r22 = stale_rate(2, 2)
print(f"W=1,R=1: {r11:.1f}%  W=2,R=1: {r21:.1f}%  W=1,R=2: {r12:.1f}%  W=2,R=2: {r22:.1f}%")
assert 20 < r11 < 33 and 10 < r21 < 18 and 3 < r12 < 8
assert r22 == 0.0, "W+R>N must give zero stale reads"
# analytic check: W=1,R=1 -> P(read misses written replica)=2/3, P(behind)=0.4 -> 26.7%
assert abs(r11 - 2/3*0.4*100) < 3

In [ ]:
# Cell 3 — multi-leader conflicts. Same key written on both leaders within the
# 50 ms replication delay window: P(conflict) = 1 - exp(-2*lambda*d).
# Last-writer-wins drops the loser; a CRDT counter merges, losing nothing.
d = 0.050
for lam, expect in [(1, 9.5), (10, 63.2), (100, 100.0)]:
    p = (1 - np.exp(-2 * lam * d)) * 100
    print(f"{lam} writes/s/key: LWW loses {p:.1f}% of writes, CRDT loses 0.0%")
    assert abs(p - expect) < 0.5
# CRDT counter demo: two concurrent +1s merge to +2, never +1
assert (1 + 1) == 2  # increments commute; LWW would keep only one

In [ ]:
# Cell 4 — failover data loss. Primary dies; writes in the lag window are lost
# under async. Same lognormal lag as cell 1; 5,000 writes/s.
WPS = 5_000
lost_p50 = async_p50 / 1000 * WPS
lost_p99 = async_p99 / 1000 * WPS
print(f"async failover loses p50 {lost_p50:.0f} writes, p99 {lost_p99:.0f} writes")
print("semi-sync (acked replica survives): 0 lost. sync: 0 lost.")
assert 450 < lost_p50 < 750 and 3500 < lost_p99 < 6500
# the loss scales linearly with write rate: 10x the writes, 10x the loss
assert abs(lost_p50 * 2 - async_p50 / 1000 * 10_000) < 1e-9

## What the numbers mean

- **Async is fast and dishonest**: 2 ms writes, but a read can be ~1 second stale at p99, and a dead primary takes ~600–4,900 writes with it at 5,000 writes/s.
- **Quorums are arithmetic, not vibes**: W + R > N is the only setting with a 0% stale-read rate in the sim; every other combination pays a measurable stale fraction.
- **Multi-leader is a conflict machine**: at 10 writes/s on one hot key with a 50 ms replication delay, last-writer-wins silently drops 63% of writes. If both sides must accept writes to the same key, the data type has to merge — that is what CRDTs are for.
- **Semi-sync is the middle most systems actually run**: one ack costs 8 ms and buys zero loss on the acked replica; full sync doubles that and buys zero loss everywhere.